# VoicePilot
__Short description__
The goal is to be able to use simple voice commands like “down”, “up”, “left”, and other keywords for different keybinds, where ultimately the model can be used on different games with similar inputs.

In [15]:
import tensorflow as tf
import tensorflow_datasets as tfds
from keras import layers, models
import keras

In [2]:
builder = tfds.builder('speech_commands')
builder.download_and_prepare()
ds = builder.as_dataset(split='train', shuffle_files=True)

2026-03-10 12:20:55.518431: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
print(ds)

<_PrefetchDataset element_spec={'audio': TensorSpec(shape=(None,), dtype=tf.int16, name=None), 'label': TensorSpec(shape=(), dtype=tf.int64, name=None)}>


In [10]:
# The dataset is recorded at 16kHz, so 1 second = 16,000 samples
TARGET_SAMPLES = 16000

def preprocess_example(example):
    # Extract the tensors from the dictionary
    audio = example['audio']
    label = example['label']

    # 1. Normalize: Convert 16-bit integers to floats between -1.0 and 1.0
    audio = tf.cast(audio, tf.float32) / 32768.0

    # 2. Fix Length: Truncate if too long, pad with zeros if too short
    audio = audio[:TARGET_SAMPLES]
    zero_padding = tf.zeros([TARGET_SAMPLES - tf.shape(audio)[0]], dtype=tf.float32)
    audio = tf.concat([audio, zero_padding], axis=0)

    # 3. Reshape: Add a channel dimension so Keras Conv1D accepts it -> (16000, 1)
    audio = tf.expand_dims(audio, axis=-1)

    # Return a tuple of (input, target) for Keras
    return audio, label

In [13]:
batch_size = 32

# Map the preprocessing function across all examples using available CPU threads
train_ds = ds.map(preprocess_example, num_parallel_calls=tf.data.AUTOTUNE)

# Optimize the pipeline for training
train_ds = (
    train_ds
    .cache()                    # Saves processed audio in memory after the first epoch
    .shuffle(10000)             # Mixes up the examples
    .batch(batch_size)          # Groups them into batches of 32
    .prefetch(tf.data.AUTOTUNE) # Prepares the next batch while the current one trains
)

# You can now print the element_spec again to see the new, clean structure:
print(train_ds.element_spec)

(TensorSpec(shape=(None, None, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))


In [ ]:
num_classes = builder.info.features['label'].num_classes

model = keras.models.Sequential([
    layers.Input(shape=(TARGET_SAMPLES, 1)),

    layers.Conv1D(16, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=4),

    layers.Conv1D(32, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=4),

    layers.Conv1D(64, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=4),

    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model directly on your processed pipeline
model.fit(train_ds, epochs=5)

 458/2673 ━━━━━━━━━━━━━━━━━━━━ 2:44 74ms/step - accuracy: 0.6239 - loss: 1.7415